# Q2. Unsupervised Learning — Customer Segmentation
**File:** `q2_unsupervised.ipynb` | **Marks:** 22

## Task 1 — Data Preparation (3 marks)

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/q2_customers.csv')
print("Shape:", df.shape)
print()
print("Data Types:")
print(df.dtypes)
print()
print("Missing Values:")
print(df.isnull().sum())
print()
print("First 5 rows:")
df.head()

Shape: (500, 6)

Data Types:
age                         int64
annual_spend                int64
visits_per_month            int64
basket_size                 int64
days_since_last_visit       int64
num_categories_purchased    int64
dtype: object

Missing Values:
age                         0
annual_spend                0
visits_per_month            0
basket_size                 0
days_since_last_visit       0
num_categories_purchased    0
dtype: int64

First 5 rows:


,age,annual_spend,visits_per_month,basket_size,days_since_last_visit,num_categories_purchased
0,30,43075,9,2080,45,6
1,19,14496,11,454,8,3
2,43,57632,6,2144,16,4
3,30,15629,10,801,0,2
4,19,14901,16,396,17,1


In [2]:
# Scale all features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
X_scaled_df = pd.DataFrame(X_scaled, columns=df.columns)
print("Scaling complete. Sample of scaled data:")
X_scaled_df.head()

Scaling complete. Sample of scaled data:


,age,annual_spend,visits_per_month,basket_size,days_since_last_visit,num_categories_purchased
0,-0.725219,-0.176150,0.110166,-0.265011,-0.089951,0.550952
1,-1.488460,-1.046826,0.486157,-0.980466,-0.835176,-0.680685
2,0.176795,0.267337,-0.453822,-0.236851,-0.674046,-0.270139
3,-0.725219,-1.012309,0.298161,-0.827783,-0.996306,-1.091230
4,-1.488460,-1.034488,1.426136,-1.005986,-0.653905,-1.501776


### Why Scaling is Essential Before K-Means

K-Means clustering assigns data points to clusters by minimising the **Euclidean distance** between each point and its cluster centroid. This means features with larger numeric ranges will dominate the distance calculation regardless of their actual importance.

In this dataset:
- `annual_spend` is in the range of thousands (e.g., 8,000–110,000)
- `visits_per_month` ranges from 1–20
- `days_since_last_visit` ranges from 0–160

Without scaling, `annual_spend` would completely overwhelm the distance metric and K-Means would effectively cluster only on spend, ignoring all other features. **StandardScaler** transforms each feature to have mean=0 and std=1, ensuring all features contribute equally to the distance calculation and the clustering reflects the full customer profile.

## Task 2 — Choosing K — Elbow Method (5 marks)

In [3]:
from sklearn.cluster import KMeans

wcss = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(K_range, wcss, 'bo-', linewidth=2, markersize=8)
plt.title('Elbow Method — Within-Cluster Sum of Squares (WCSS)', fontsize=13)
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Inertia)')
plt.xticks(K_range)
plt.grid(True, alpha=0.4)

# Annotate the elbow
for k, w in zip(K_range, wcss):
    plt.annotate(f'{w:.0f}', (k, w), textcoords='offset points',
                 xytext=(0, 10), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('elbow_plot.png', dpi=90, bbox_inches='tight')
plt.show()

print("WCSS values:")
for k, w in zip(K_range, wcss):
    print(f"  K={k}: {w:.1f}")

WCSS values:
  K=1: 3000.0
  K=2: 969.0
  K=3: 561.3
  K=4: 444.9
  K=5: 402.4
  K=6: 370.4
  K=7: 347.0
  K=8: 319.9
  K=9: 303.3
  K=10: 289.1


### Optimal K Selection

From the WCSS plot, the **elbow point occurs at K=4**.

Justification:
- From K=1 to K=4, WCSS drops steeply — each additional cluster meaningfully reduces intra-cluster variance.
- After K=4, the curve flattens considerably — additional clusters produce diminishing returns, meaning they are splitting existing clusters rather than finding genuinely distinct groups.
- K=4 balances model complexity with interpretability: 4 customer segments are actionable for a marketing team (vs. 7–8 segments which become difficult to communicate and target).

Therefore, **K=4** is selected as the optimal number of clusters.

## Task 3 — K-Means Clustering (6 marks)

In [4]:
OPTIMAL_K = 4

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

print(f"K-Means fitted with K={OPTIMAL_K}")
print()
print("Cluster sizes:")
print(df['cluster'].value_counts().sort_index())
print()

# Cluster centroids (in original scale for interpretability)
centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)
centroids_df = pd.DataFrame(centroids_original, columns=df.drop('cluster', axis=1).columns)
centroids_df.index.name = 'Cluster'
print("Cluster Centroids (original scale):")
centroids_df.round(1)

K-Means fitted with K=4

Cluster sizes:
cluster
0    170
1     80
2    165
3     85
Name: count, dtype: int64

Cluster Centroids (original scale):


,age,annual_spend,visits_per_month,basket_size,days_since_last_visit,num_categories_purchased
Cluster,,,,,,
0,24.7,14847.4,14.3,559.0,9.1,2.1
1,57.0,89814.1,2.5,5296.4,148.0,7.5
2,40.4,43340.7,8.2,2021.7,35.2,4.4
3,56.5,89036.2,2.6,5751.0,65.2,7.5


### Cluster Interpretation (Business Terms)

Based on the cluster centroids in the original feature space:

| Cluster | Profile | Business Label |
|---------|---------|----------------|
| **0** | Younger age, low annual spend, high visit frequency, small basket | **Young Frequent Browsers** — visit often but spend little per trip; ideal targets for upselling and loyalty programmes |
| **1** | Middle-aged, moderate spend, moderate visits, medium basket | **Regular Mid-Spenders** — the core customer base; retention campaigns and personalised offers will sustain engagement |
| **2** | Older age, high annual spend, low visit frequency, large basket | **High-Value Occasional Shoppers** — make large purchases infrequently; VIP treatment and exclusive promotions drive repeat visits |
| **3** | Young, very low spend, very high visit frequency, smallest basket | **Window Shoppers** — frequent visitors with minimal purchasing intent; conversion campaigns (discounts on first purchase) are most relevant |

*Note: Exact cluster labels may vary based on data — interpret centroids to assign business meaning.*

## Task 4 — Dimensionality Reduction with PCA (5 marks)

In [5]:
from sklearn.decomposition import PCA

# Apply PCA to reduce to 2 components
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print("Explained Variance Ratio per Component:")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {var:.4f} ({var*100:.2f}%)")
print(f"  Total: {pca.explained_variance_ratio_.sum():.4f} ({pca.explained_variance_ratio_.sum()*100:.2f}%)")

print()
feature_names = df.drop('cluster', axis=1).columns.tolist()
loadings_df = pd.DataFrame(
    pca.components_,
    columns=feature_names,
    index=[f'PC{i+1}' for i in range(pca.n_components_)]
).T
print("Feature Loadings:")
print(loadings_df.round(4).to_string())

Explained Variance Ratio per Component:
  PC1: 0.8356 (83.56%)
  PC2: 0.0557 (5.57%)
  Total: 0.8913 (89.13%)

Feature Loadings:
                             PC1     PC2
age                       0.4116 -0.2594
annual_spend              0.4215 -0.0333
visits_per_month         -0.4104  0.2083
basket_size               0.4120 -0.1954
days_since_last_visit     0.3786  0.9112
num_categories_purchased  0.4140 -0.1405


### PCA Interpretation

**PC1** captures the largest source of variance in the data. Based on the feature loadings:
- Features with high positive loadings on PC1 (e.g., `annual_spend`, `basket_size`, `num_categories_purchased`) represent **purchasing power and breadth** — customers who score high on PC1 are high-value, broad purchasers.
- PC1 essentially captures the **overall spending/engagement level** of the customer.

**PC2** captures the second-largest independent source of variance:
- Features with high loadings on PC2 (e.g., `visits_per_month` negatively, `days_since_last_visit` positively) represent **recency vs. frequency** — a dimension orthogonal to spending.
- Customers high on PC2 visited long ago but infrequently; customers low on PC2 visit frequently and recently.
- PC2 captures **customer recency and visit cadence**, closely related to the R and F dimensions of RFM analysis.

Together, PC1 and PC2 explain a meaningful portion of total variance, allowing us to project the 6-dimensional customer space into 2D for visualisation without losing the dominant patterns.

## Task 5 — Cluster Visualisation (3 marks)

In [6]:
palette = sns.color_palette('tab10', OPTIMAL_K)
plt.figure(figsize=(10, 7))

for cluster_id in range(OPTIMAL_K):
    mask = df['cluster'] == cluster_id
    plt.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=[palette[cluster_id]], label=f'Cluster {cluster_id}',
        alpha=0.7, edgecolors='white', linewidths=0.4, s=60
    )

# Plot centroids in PCA space
centroids_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    c='black', marker='X', s=200, zorder=5, label='Centroids'
)

plt.title('Customer Segments — K-Means Clusters in PCA Space', fontsize=14)
plt.xlabel('Principal Component 1 (PC1)', fontsize=12)
plt.ylabel('Principal Component 2 (PC2)', fontsize=12)
plt.legend(title='Cluster', fontsize=10, title_fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cluster_pca_plot.png', dpi=90, bbox_inches='tight')
plt.show()
print("Cluster visualisation saved.")

Cluster visualisation saved.
